# Sprint 1 - PixelPaw CLI
### วิชา CP352301 Script Programming

**ทีม:** [ชื่อ Planner] (Planner/PM) · ศุภชัย คนเพียร (Coder) · [ชื่อ Debugger] (Debugger/QA)
*(กรณีทำคนเดียว: ศุภชัย คนเพียร หมุนเวียนทั้ง 3 บทบาทในสัปดาห์เดียวกัน)*

**Repository:** https://github.com/suphachaikh-creator/Final-Project_G11_Sec2_Script_Programming

**Sprint:** Sprint 1 - Front-End App Dev (สัปดาห์ที่ 12)

หมายเหตุ: Sprint 1 เน้นการออกแบบส่วนปฏิสัมพันธ์กับผู้ใช้ (UI/CLI), การจัดการเมนู และการตรวจสอบความถูกต้องของข้อมูลนำเข้า (Input Validation) โดย **ยังไม่มีการเชื่อมต่อ External API และยังไม่บันทึกข้อมูลลงไฟล์ถาวร** ซึ่งเป็นแผนของ Sprint ถัดไป

---
## สารบัญ
0. [สรุปความก้าวหน้าของงาน](#0)
1. [Planning - PLAN.md](#1)
2. [Architecture ปัจจุบัน](#2)
3. [Source Code](#3)
4. [Demo / วิธีใช้งาน](#4)
5. [Test Report - Edge Case Testing](#5)
6. [Retrospective - Wow! & Whoops!](#6)
7. [แผนต่อยอดเป็น Sprint 2](#7)
8. [บทบาทและ Self-assessment ตามเกณฑ์](#8)

<a id="0"></a>
## 0. สรุปความก้าวหน้าของงาน (Sprint Progress Summary)

- [x] ออกแบบโครงสร้างระบบและนิยาม Definition of Done 7 ข้อ ใน `PLAN.md`
- [x] พัฒนาชุดฟังก์ชันหลัก `display_welcome_message` · `get_command_input` · `prompt_until_valid` · `run_app`
- [x] แยกโค้ดเป็น 4 โมดูลตามหลัก Modular (`app` / `ui` / `validators` / `mock_data`)
- [x] ดักจับข้อผิดพลาดกรณีผู้ใช้ป้อนข้อมูลผิดรูปแบบด้วย `try-except ValueError`
- [x] ดักจับกรณีสตรีมอินพุตปิดกลางคันด้วย `try-except EOFError`
- [x] ทดสอบเคสขอบเขต 11 เคส และแปลงเป็น automated test 36 เคสด้วย `pytest`
- [x] ตรวจมาตรฐานโค้ด PEP 8 ด้วย `flake8` ผ่านโดยไม่มีข้อผิดพลาด
- [x] ตั้งค่า CI Pipeline บน GitHub Actions ให้รัน lint และ test อัตโนมัติทุก push
- [ ] ส่งมอบงานผ่าน Pull Request พร้อมสรุป Wow! / Whoops!

**ลิงก์ Repository:** https://github.com/suphachaikh-creator/Final-Project_G11_Sec2_Script_Programming

**ลิงก์ Pull Request:** `[แปะลิงก์ PR หลังเปิด]`

In [ ]:
"""เซลล์ตั้งค่า - รันเซลล์นี้ก่อนเสมอ

โน้ตบุ๊กเก็บอยู่ที่ Sprint1/notebook/ ส่วนซอร์สโค้ดอยู่ที่ Sprint1/code/
เซลล์นี้จะย้ายไปทำงานที่โฟลเดอร์ code เพื่อให้ import แพ็กเกจ src ได้ถูกต้อง
"""
import os
import subprocess
import sys
from pathlib import Path

SPRINT_NAME = "Sprint1"


def find_code_dir(start):
    """ค้นหาโฟลเดอร์ code ให้เจอ ไม่ว่าจะเปิดโน้ตบุ๊กจากตำแหน่งใด"""
    candidates = [
        start.parent / "code",             # เปิดจาก Sprint1/notebook/ (ปกติ)
        start / "code",                    # เปิดจาก Sprint1/
        start / SPRINT_NAME / "code",      # เปิดจาก root ของ repo
        start,                             # อยู่ในโฟลเดอร์ code อยู่แล้ว
    ]
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "tests").is_dir():
            return candidate.resolve()
    raise RuntimeError(f"ไม่พบโฟลเดอร์ code ของ {SPRINT_NAME}")


CODE_DIR = find_code_dir(Path.cwd())
os.chdir(CODE_DIR)

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))


def show_source(relative_path):
    """แสดงซอร์สโค้ดจริงจากไฟล์ เพื่อไม่ให้รายงานหลุดจากโค้ดใน repo"""
    print(f"# ===== {relative_path} =====")
    print((CODE_DIR / relative_path).read_text(encoding="utf-8"))


print("โฟลเดอร์โค้ด:", CODE_DIR)
print("ไฟล์ใน src/:", sorted(p.name for p in (CODE_DIR / "src").glob("*.py")))

<a id="1"></a>
## 1. Planning

**ภารกิจ Planner:**

**ขอบเขตระบบ (Scope):** แอปจำลองการเลี้ยงสัตว์เสมือนจริงแบบ Command-Line เขียนด้วย Python ล้วน แยกเป็นโมดูลใน `src/` ทำงานทั้งหมดใน memory ระหว่างโปรแกรมรัน **ยังไม่เชื่อมต่อ API ภายนอกและยังไม่บันทึกข้อมูลลงไฟล์** ตามกติกาของสปรินต์ Front-End

**เมนูหลัก 4 ตัวเลือก:**
- 1) รับเลี้ยงน้องแมว - เลือกสายพันธุ์จากข้อมูลจำลอง ตั้งชื่อและอายุ เริ่มค่าสถานะที่ 50/50/50
- 2) ดูสถานะน้องแมว - แสดงค่าสถานะที่เก็บอยู่ใน memory
- 3) เมนูดูแลน้องแมว - ให้อาหาร / เล่น / นอนพัก (Sprint 1 เป็นโครงหน้าจอ ยังไม่คำนวณค่าจริง)
- 4) ออกจากโปรแกรม

**Definition of Done (DoD):**

| # | เงื่อนไข | ตรวจด้วยเทสต์ |
|---|---|---|
| 1 | โปรแกรมรันผ่าน Terminal (CLI) และแสดงเมนูวนลูปได้ ยกเว้นเลือกออก | `test_app_flow.py` |
| 2 | คำสั่ง `quit`, `QUIT`, `Quit`, `exit`, `ออก` ออกจากโปรแกรมได้ทุกหน้าจอ ไม่ว่าตัวพิมพ์เล็กหรือใหญ่ | `test_quit_exits_immediately` |
| 3 | ป้อนเมนูผิดหรือป้อนข้อความที่ไม่ใช่ตัวเลข ต้องแจ้งเตือนและกลับมารับคำสั่งใหม่ โดยโปรแกรมไม่ crash | `test_invalid_menu_choice_does_not_crash` |
| 4 | ต้องรับเลี้ยงน้องแมวก่อน จึงจะดูสถานะหรือเข้าเมนูดูแลได้ ถ้ายังไม่มีต้องแจ้งเตือน ไม่ crash | `test_status_requires_adoption_first` |
| 5 | ชื่อน้องแมวห้ามว่างเปล่า และยาวไม่เกิน 20 ตัวอักษร | `TestValidatePetName` |
| 6 | อายุต้องเป็นจำนวนเต็มบวก (0-30) ปฏิเสธค่าติดลบและทศนิยม | `TestValidateAge` |
| 7 | ค่าสถานะทุกตัวถูกจำกัดขอบเขตไว้ที่ 0-100 เสมอ ไม่ล้นบวกไม่ติดลบ | `TestClampStat` |

**ภารกิจ Coder/Debugger:** ทบทวนสเปกกับ Planner จนเข้าใจตรงกันก่อนเริ่มเขียนโค้ด ยืนยันแล้วว่าโครงสร้างโมดูลตรงกับที่ตกลงไว้ (เอกสารฉบับเต็มอยู่ใน `Sprint1/PLAN.md`)

<a id="2"></a>
## 2. Architecture ปัจจุบัน (CLI, Front-End เท่านั้น)

```
Terminal
   |
   v
main.py                    entry point + บังคับ UTF-8 บน Windows Console
   |
   v
src/app.py                 Application Layer
   |  +- run_app()              while True + try/except ValueError
   |  +- get_command_input()    .strip() + ดัก EOFError
   |  +- prompt_until_valid()   ถามซ้ำจนอินพุตผ่าน
   |  +- adopt_pet()            เลือกสายพันธุ์ -> ตั้งชื่อ -> ระบุอายุ
   |  +- run_care_menu()        เมนูดูแล (ยังไม่คำนวณค่า)
   |
   +--> src/ui.py          Presentation Layer  (print เท่านั้น ไม่มีตรรกะ)
   +--> src/validators.py  Validation Layer    (ไม่มี input()/print() จึงเทสต์ตรงได้)
   +--> src/mock_data.py   Mock Data (RAM)     (แทนผลลัพธ์ API ของ Sprint 2)
```

**Pattern ที่ยึด - UI/Core separation:** `validators.py` ไม่รู้จัก `input()` และ `ui.py` ไม่คำนวณตรรกะใดๆ ทำให้ทดสอบแต่ละชั้นแยกกันได้โดยไม่ต้องจำลองหน้าจอ

**สิ่งที่ยังไม่มีในเวอร์ชันนี้ (ยกไปเป็นงานของ Sprint ถัดไป):**

| รายการ | เหตุผลที่ยังไม่ทำใน Sprint 1 | ปลายทาง |
|---|---|---|
| เชื่อม API ดึงสายพันธุ์จริง | สปรินต์นี้เป็น Front-End ห้ามเรียก API จริง | Sprint 2 |
| บันทึกและโหลดไฟล์ JSON | ห้ามเขียนไฟล์จริง ใช้ข้อมูลใน memory | Sprint 2 |
| ตรรกะ feed / play / rest และคลาส `Pet` | เป็น business logic ของสปรินต์ถัดไป | Sprint 2 |
| Time Decay ตามเวลาจริง | ต้องมี persistence เก็บ timestamp ก่อน | Sprint 3 |

<a id="3"></a>
## 3. Source Code

**ภารกิจ Coder:** โค้ดกระชับ ตั้งชื่อตัวแปรสื่อความหมาย มี Docstring อธิบายทุกฟังก์ชัน แยกไฟล์ตามหลัก Modular (ui / validators / mock_data / app)

เซลล์ด้านล่างอ่านซอร์สจากไฟล์จริงใน `src/` โดยตรง ทำให้รายงานตรงกับโค้ดใน repo เสมอ

In [ ]:
show_source("src/validators.py")

In [ ]:
show_source("src/mock_data.py")

In [ ]:
show_source("src/ui.py")

In [ ]:
show_source("src/app.py")

<a id="4"></a>
## 4. Demo / วิธีใช้งาน

### รันจริงผ่าน Terminal

```bash
cd Sprint1/code
python main.py
```

- เลือกเมนู 1 เพื่อรับเลี้ยงน้องแมว: เลือกสายพันธุ์ ตั้งชื่อ ระบุอายุ
- เลือกเมนู 2 เพื่อดูสถานะปัจจุบัน
- เลือกเมนู 3 เพื่อเข้าเมนูดูแล (ให้อาหาร / เล่น / นอนพัก)
- เลือกเมนู 4 หรือพิมพ์ `quit` เพื่อออกจากโปรแกรม

### รันในโน้ตบุ๊กแบบจำลองการพิมพ์

เซลล์ถัดไปสาธิตการทำงานจริงของ `run_app()` โดยป้อนคำสั่งตามสคริปต์ที่กำหนดไว้ล่วงหน้า (ใช้วิธีเดียวกับ integration test) จึงรันในโน้ตบุ๊กได้โดยไม่ต้องพิมพ์ตอบทีละบรรทัด

In [ ]:
"""สาธิตการทำงานจริง โดยแทน input() ด้วยลำดับคำสั่งที่กำหนดไว้ล่วงหน้า"""
import builtins

from src.app import run_app


def scripted_input(commands):
    """คืนฟังก์ชันแทน input() ที่ป้อนคำสั่งตามลำดับ และพิมพ์คำตอบของผู้ใช้ออกหน้าจอ"""
    queue = list(commands)

    def _fake_input(prompt=""):
        if not queue:
            raise EOFError
        answer = queue.pop(0)
        print(f"{prompt}{answer}")
        return answer

    return _fake_input


DEMO_COMMANDS = [
    "1",      # รับเลี้ยงน้องแมว
    "1",      # เลือกสายพันธุ์ Siamese
    "มิว",    # ตั้งชื่อ
    "3",      # อายุ 3 ปี
    "3",      # เข้าเมนูดูแล
    "1",      # ให้อาหาร
    "4",      # กลับเมนูหลัก
    "quit",   # ออกจากโปรแกรม
]

original_input = builtins.input
builtins.input = scripted_input(DEMO_COMMANDS)
try:
    run_app()
finally:
    builtins.input = original_input

In [ ]:
"""สาธิตการดักอินพุตผิดรูปแบบ โปรแกรมต้องเตือนแล้ววนกลับเมนู ไม่ crash"""
ERROR_COMMANDS = [
    "abc",    # ไม่ใช่ตัวเลข
    "99",     # เกินช่วงเมนู
    "2",      # ดูสถานะทั้งที่ยังไม่มีน้องแมว
    "QUIT",   # ออกด้วยตัวพิมพ์ใหญ่
]

builtins.input = scripted_input(ERROR_COMMANDS)
try:
    run_app()
finally:
    builtins.input = original_input

In [ ]:
"""รันแบบโต้ตอบจริง (เอาเครื่องหมาย # ออกเพื่อพิมพ์คำสั่งเอง)"""
# from src.app import run_app
# run_app()

<a id="5"></a>
## 5. Test Report - Edge Case Testing

**ภารกิจ Debugger:** ทดสอบอย่างเป็นระบบตามรูปแบบ Observation / Expected / Actual แล้วแปลงทุกเคสเป็น **automated test จริงด้วย pytest** ในโฟลเดอร์ `tests/`

| รายการทดสอบ | อินพุตที่ใช้ | ผลลัพธ์ที่คาดหวัง (Expected) | ผลการทดสอบจริง (Actual) | สถานะ |
|---|---|---|---|---|
| ชื่อน้องแมวว่างเปล่า | `"   "` | แจ้งเตือน ไม่ยอมรับ ให้กรอกใหม่ | `ValueError("ชื่อน้องแมวห้ามเว้นว่าง")` และวนถามซ้ำ | **PASSED** |
| ชื่อยาวเกินกำหนด | `"x" * 21` | แจ้งเตือนว่าเกิน 20 ตัวอักษร | ปฏิเสธและวนถามซ้ำ | **PASSED** |
| อายุไม่ใช่ตัวเลข | `"สามปี"` | แจ้งเตือนว่าต้องกรอกตัวเลข | `ValueError("อายุต้องเป็นจำนวนเต็มบวกเท่านั้น")` | **PASSED** |
| อายุติดลบ | `"-5"` | ปฏิเสธ ไม่รับค่าติดลบ | `.isdigit()` คืน False จึงปฏิเสธก่อนแปลงเป็น int | **PASSED** |
| อายุเป็นทศนิยม | `"2.5"` | ปฏิเสธ ต้องเป็นจำนวนเต็ม | ปฏิเสธถูกต้อง | **PASSED** |
| อายุเกินความเป็นจริง | `"99"` | แจ้งเตือนว่าต้องไม่เกิน 30 ปี | ปฏิเสธถูกต้อง | **PASSED** |
| ดูสถานะก่อนรับเลี้ยง | เลือกเมนู `2` ทันที | เตือนให้รับเลี้ยงก่อน ไม่ crash | แสดง `ยังไม่มีน้องแมว กรุณาเลือกเมนู 1 ก่อน` | **PASSED** |
| เมนูไม่ใช่ตัวเลข 1-4 | `"abc"`, `"99"` | แจ้งเตือน ไม่ทำให้โปรแกรม crash | แสดง `[X]` สองครั้งแล้ววนกลับเมนู | **PASSED** |
| ความไวต่อตัวพิมพ์เล็ก-ใหญ่ของคำสั่งออก | `"quit"`, `"QUIT"`, `"Quit"`, `" quit "` | ออกจากโปรแกรมได้ทุกกรณี | `normalize_command()` ใช้ `.strip().lower()` ก่อนเทียบ | **PASSED** |
| ค่าสถานะเกินขอบเขต | `clamp_stat(150)`, `clamp_stat(-10)` | บีบกลับเป็น 100 และ 0 | `clamp_stat()` ล็อกช่วง 0-100 ถูกต้อง | **PASSED** |
| สตรีมอินพุตปิดกลางคัน | ป้อนข้อมูลผ่าน pipe จนหมด | ถือว่าผู้ใช้สั่งออก ไม่โยน `EOFError` ออกมา | `get_command_input()` ดัก `EOFError` แล้วคืน `"quit"` | **PASSED** |

### รันชุดทดสอบอัตโนมัติจริง

In [ ]:
"""รันชุดทดสอบจริงด้วย pytest (ชุดเดียวกับที่ GitHub Actions ใช้ตรวจ)"""
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-v", "--no-header"],
    capture_output=True, text=True, encoding="utf-8", cwd=str(CODE_DIR),
)
print(result.stdout[-4000:])
print("exit code:", result.returncode)

### ตรวจมาตรฐานโค้ด (PEP 8)

ขั้นตอนเดียวกับที่ GitHub Actions ใช้ใน job `Lint & Test (Sprint1)`

In [ ]:
"""ตรวจมาตรฐานโค้ด PEP 8 ด้วย flake8 (ขั้นตอน Lint ของ CI)"""
result = subprocess.run(
    [sys.executable, "-m", "flake8", "."],
    capture_output=True, text=True, encoding="utf-8", cwd=str(CODE_DIR),
)
print(result.stdout or "flake8: ไม่พบข้อผิดพลาด (0 issues)")
print("exit code:", result.returncode)

<a id="6"></a>
## 6. Retrospective - Wow! & Whoops!

**Wow! (ส่วนที่ทำได้ดี):**
- แยกไฟล์ตามหน้าที่ชัดเจน (`ui` / `validators` / `mock_data` / `app`) ทำให้ `validators.py` ไม่มี `input()` และ `print()` เลย จึงเขียน unit test ได้ตรงๆ ไม่ต้องจำลองหน้าจอ
- รวมการถามซ้ำไว้ที่ `prompt_until_valid()` จุดเดียว ทุกช่องกรอกจึงได้พฤติกรรม "ผิดแล้วถามใหม่ และออกได้ตลอด" เหมือนกันหมด โดยไม่ต้องเขียนลูปซ้ำในทุกฟังก์ชัน
- มี `clamp_stat()` เป็นจุดเดียวที่ควบคุมขอบเขตค่าสถานะ ป้องกันค่าหลุดช่วง 0-100 จากทุกฟังก์ชันที่เรียกใช้
- เทสต์ครอบคลุมทั้งระดับฟังก์ชัน (`test_validators.py`) และระดับการไหลของหน้าจอ (`test_app_flow.py`) รวม 36 เคส และผูกเข้ากับ CI แล้ว

**Whoops! (ปัญหาที่พบและแนวทางแก้ไข):**
- **ทำงานเกินขอบเขตสปรินต์** เวอร์ชันแรกเขียนทั้งการเรียก API จริงและการบันทึกไฟล์ JSON ไว้ใน Sprint 1 ซึ่งผิดกติกาที่กำหนดว่าสปรินต์นี้เป็น Front-End เท่านั้น แก้ไขโดยยกไฟล์ทั้งหมดออกไปเป็นงานของ Sprint 2 แล้วแทนที่ด้วย `mock_data.py`
- **ตรวจอายุติดลบไม่ผ่านตั้งแต่แรก** `int("-5")` ไม่ raise `ValueError` ทำให้ค่าติดลบหลุดเข้าระบบ แก้ไขด้วยการตรวจ `.isdigit()` ก่อนแปลงเป็น int แทนการพึ่ง try-except อย่างเดียว
- **ภาษาไทยเพี้ยนบน Windows Console** ตอนรันแบบ pipe ชื่อที่พิมพ์เป็นภาษาไทยกลายเป็นอักขระขยะ เพราะ stdin ใช้ code page 874 แก้ไขด้วยการ `reconfigure(encoding="utf-8")` ทั้ง `sys.stdin` และ `sys.stdout` ใน `main.py`
- **โปรแกรมพังเมื่อสตรีมอินพุตหมด** `input()` โยน `EOFError` ซึ่ง `try-except ValueError` ดักไม่ได้ แก้ไขโดยดัก `EOFError` ใน `get_command_input()` แล้วถือว่าผู้ใช้สั่งออกจากโปรแกรม
- ข้อมูลทั้งหมดยังอยู่ใน memory หากปิดโปรแกรมสถานะจะหายทันที เป็นประเด็นที่แก้ใน Sprint 2 ด้วย Data Persistence

<a id="7"></a>
## 7. แผนต่อยอดเป็น Sprint 2 (Back-End + Persistence)

เป้าหมายคือเพิ่ม Business Logic เต็มรูปแบบให้ PixelPaw โดยคงหน้าจอ CLI ของ Sprint 1 ไว้เป็นแกนหลัก แล้วเพิ่มการเชื่อมต่อ API ภายนอก การบันทึกข้อมูลถาวร และ Time Decay ตามเวลาจริง

### การเชื่อมต่อ API และการเก็บข้อมูล

| รายการ | รายละเอียด |
|---|---|
| API ที่ใช้ | Gemini API สร้างข้อมูลสายพันธุ์แมว (breed / country / temperament / stubbornness_score) |
| การจัดการข้อผิดพลาด | ครอบ try-except รอบการเรียก API และตัด markdown code fence ก่อน `json.loads()` |
| Data Persistence | ไฟล์ JSON (`data/pet_state.json`) เก็บสถานะปัจจุบันของน้องแมว |
| Time Decay | เก็บ `last_updated` (timestamp) ใน state แล้วลดค่าสถานะตามชั่วโมงที่ผ่านไปจริงตอนเปิดโปรแกรม |

### โครงสร้างโค้ดที่วางแผนไว้สำหรับ Sprint 2

```
src/pet.py           -> คลาส Pet: feed/play/rest + to_dict/from_dict
src/api_client.py    -> เรียก Gemini API + แปลงผลลัพธ์เป็น JSON
src/data_store.py    -> save_game / load_game (os.path.exists guard) / fetch_new_pets
src/game.py          -> ลูปการเลี้ยงที่เชื่อมตรรกะจริง
src/app.py           -> เมนูหลักเชื่อม API และไฟล์เซฟ
main.py              -> entry point
tests/               -> unit test แบบ pytest ของแต่ละโมดูล
```

### สิ่งที่ต้องแก้ก่อนและระหว่าง Sprint 2
- เปลี่ยนสถานะจาก `dict` ใน memory ไปเป็นอ็อบเจกต์ `Pet` เพื่อรองรับเกณฑ์ OOP (ออกแบบไว้แล้วในแผน Sprint 2)
- ทำให้เทสต์ของ Sprint 2 รันได้โดย **ไม่ต้องใช้ API Key และไม่ต้องต่ออินเทอร์เน็ต** เพื่อให้ CI ผ่าน (ใช้ `monkeypatch` และ `tmp_path`)
- เชื่อมหน้าจอของ Sprint 1 เข้ากับตรรกะของ Sprint 2 ใน Sprint 3 (Full-Stack)

<a id="8"></a>
## 8. บทบาทและ Self-assessment ตามเกณฑ์การประเมิน

| บทบาท | ผู้รับผิดชอบ | งานหลักใน Sprint 1 |
|---|---|---|
| Planner / Team Leader | [ชื่อนักศึกษา] | เขียนสเปกใน `PLAN.md` กำหนดขอบเขตเมนู นิยาม Definition of Done |
| Coder | ศุภชัย คนเพียร | พัฒนาโมดูล `app` / `ui` / `validators` / `mock_data` ตามสเปกที่ตกลงกันไว้ |
| Debugger / QA | [ชื่อนักศึกษา] | ทดสอบ edge cases แปลงเป็น pytest 36 เคส สรุป Retrospective |

### Self-assessment เทียบกับเกณฑ์ Role-Based Grading Rubric

| หัวข้อ | ระดับที่ประเมินตนเอง | เหตุผล |
|---|---|---|
| Planner - การวางแผนและกำหนดขอบเขตงาน | [ระบุ] | ระบุเมนู 4 ตัวเลือกและ DoD 7 ข้อใน `PLAN.md` ก่อนเริ่มเขียนโค้ด พร้อมระบุชัดว่าอะไร **ไม่** อยู่ในสปรินต์นี้ |
| Planner - Definition of Done | [ระบุ] | ครอบคลุมทั้ง happy path และกรณี error และผูกแต่ละข้อเข้ากับเทสต์ที่ตรวจจริง |
| Coder - การจัดโครงสร้างโค้ดและโมดูล | [ระบุ] | แยก 4 โมดูลตามหน้าที่ ยึด UI/Core separation และผ่าน flake8 โดยไม่มีข้อผิดพลาด |
| Coder - การจัดการอินพุตและสถานะโปรแกรม | [ระบุ] | ใช้ `while True` + `try/except ValueError` + `prompt_until_valid()` + `clamp_stat()` ป้องกันค่าเพี้ยน |
| Debugger - การทดสอบเคสขอบเขต | [ระบุ] | 11 เคสขอบเขต แปลงเป็น automated test 36 เคส ครอบคลุมทั้ง validator และ flow ของหน้าจอ |
| Debugger - Exception Handling | [ระบุ] | ดักทั้ง `ValueError` (อินพุตผิดรูปแบบ) และ `EOFError` (สตรีมอินพุตปิด) จึงไม่มีทางที่โปรแกรมจะ crash จากอินพุต |
| DevOps - CI/CD | [ระบุ] | `.github/workflows/ci.yml` รัน flake8 และ pytest อัตโนมัติทุก push และ pull request |

### เอกสารอ้างอิง / Prompt ที่ใช้ในการพัฒนา
- Prompt สำหรับสร้าง Project Pitch และ `PLAN.md` จาก Final Term Project Requirements
- Prompt ตรวจสอบว่างานใดเกินขอบเขต Sprint 1 และควรย้ายไป Sprint 2
- Prompt ออกแบบ GitHub Actions workflow ให้ตรวจสองสปรินต์แบบ matrix
- Prompt แปลง assert-based test ใน Colab ให้เป็นชุดทดสอบ pytest จริงในโฟลเดอร์ `tests/`